In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

class CustomSegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = sorted(os.listdir(image_dir))
        
    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)
        
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        
        if self.transform:
            image = self.transform(image)
            mask = transforms.ToTensor()(mask) 
            mask = (mask > 0).float()
            
        return image, mask


def calculate_metrics(pred_mask, true_mask, epsilon=1e-6):
    pred_mask = pred_mask.view(-1).byte()
    true_mask = true_mask.view(-1).byte()

    intersection = (pred_mask & true_mask).float().sum()
    union = (pred_mask | true_mask).float().sum()
    
    iou = (intersection + epsilon) / (union + epsilon)
    dice = (2.0 * intersection + epsilon) / (pred_mask.float().sum() + true_mask.float().sum() + epsilon)
    
    return iou.item(), dice.item()

def visualize_overlays(image_tensor, true_mask_tensor, pred_mask_tensor, iou_score, alpha=0.4):

    image = image_tensor[0].cpu().numpy().transpose(1, 2, 0)
    image = np.clip(image, 0, 1)
    

    true_mask = true_mask_tensor[0].cpu().numpy().squeeze()
    pred_mask = pred_mask_tensor[0].cpu().numpy().squeeze()


    gt_overlay = np.copy(image)
    gt_overlay[true_mask == 1] = gt_overlay[true_mask == 1] * (1 - alpha) + np.array([0, 1, 0]) * alpha


    pred_overlay = np.copy(image)
    pred_overlay[pred_mask == 1] = pred_overlay[pred_mask == 1] * (1 - alpha) + np.array([1, 0, 0]) * alpha


    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    axes[1].imshow(gt_overlay)
    axes[1].set_title("Ground Truth (Green)")
    axes[1].axis('off')

    axes[2].imshow(pred_overlay)
    axes[2].set_title(f"Prediction (Red) - IoU: {iou_score * 100:.1f}%")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()


def evaluate_model(model, test_loader, device='cuda', max_visualizations=5):
    print(f"Moving model to {device} and setting to evaluation mode...")
    model.to(device)
    model.eval()
    
    total_iou = 0.0
    total_dice = 0.0
    num_batches = len(test_loader)
    visuals_shown = 0
    
    print(f"Starting evaluation on {num_batches} test images...")
    
    with torch.no_grad():
        for images, true_masks in tqdm(test_loader, desc="Evaluating"):
            images = images.to(device)
            true_masks = true_masks.to(device)
            
            raw_predictions = model(images)
            probabilities = torch.sigmoid(raw_predictions)
            binary_predictions = (probabilities > 0.5).float()
            
            iou, dice = calculate_metrics(binary_predictions, true_masks)
            
            total_iou += iou
            total_dice += dice
            

            if visuals_shown < max_visualizations:
                visualize_overlays(images, true_masks, binary_predictions, iou)
                visuals_shown += 1

    avg_iou = total_iou / num_batches
    avg_dice = total_dice / num_batches
    
    print("\n--- Final Test Set Results ---")
    print(f"Average IoU:   {avg_iou * 100:.2f}%")
    print(f"Average Dice:  {avg_dice * 100:.2f}%")
    print("------------------------------")
    
    return avg_iou, avg_dice


if __name__ == "__main__":
    IMAGE_DIR = "./data/test/images/"
    MASK_DIR = "./data/test/masks/"
    MODEL_WEIGHTS_PATH = "./best_model.pth"
    
    image_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
     
    ])
    
    test_dataset = CustomSegmentationDataset(
        image_dir=IMAGE_DIR, 
        mask_dir=MASK_DIR, 
        transform=image_transform
    )
    
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    # model = YourSegmentationModel()
    # model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location='cpu'))
    
    active_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # evaluate_model(model, test_loader, device=active_device)